# Leakage: cómo te engañas sin querer (y crees que tu modelo es genial)

**Facsímil 8 · La ciencia de los datos** — capítulos 2 y 3
(calidad de datos y *leakage*; *splits*, muestreo y medir sin engañarse).

Casi todos los «modelos increíbles» que fracasan al desplegarse tenían **fuga de datos** (*leakage*):
información del test —o del futuro— que se coló en el entrenamiento. Es el error más traicionero del
*machine learning* porque **no da ningún error**: da *buenas noticias falsas*. En este cuaderno lo
provocas a propósito, ves un modelo presumir de un acierto altísimo sobre datos donde **no hay nada que
aprender**, descubres de dónde sale el truco, y lo arreglas para ver el número real, el del puro azar.
Después recorres los otros disfraces de la fuga (la respuesta colada en una columna, el escalado, el
tiempo, las filas duplicadas) y aprendes a olerlos. Esa nariz vale más que cualquier algoritmo.

### Qué vas a aprender
- Qué es el *leakage* y por qué es silencioso: no rompe nada, infla resultados.
- A provocar el caso canónico: **seleccionar features mirando todo el dataset** antes de partir.
- A hacerlo **bien**: separar el test antes de tocar nada y meter cada decisión dentro de la validación.
- Los disfraces de la fuga: *target leakage*, escalado, fuga temporal y filas duplicadas, cada uno con
  su demostración y cómo cazarlos.
- Cómo el problema **crece con el número de columnas** y por qué un resultado redondo debe darte miedo.

### Cuánto cuesta
Unos 20 minutos. CPU, sin claves, sin instalar nada raro.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
np.random.seed(0)

# 120 muestras, 1000 columnas de RUIDO puro, y una etiqueta ALEATORIA: no hay
# ninguna relacion entre X e y. Si un modelo "acierta" aqui, es trampa segura.
X = np.random.normal(size=(120, 1000))
y = np.random.randint(0, 2, 120)
print("Datos:", X.shape, "-> 1000 columnas de ruido y una etiqueta al azar.")
print("No hay senal que aprender: el mejor modelo posible deberia acertar el 50% (una moneda).")
print("Cualquier acierto claramente por encima del 50% sera, por definicion, una ilusion.")


## 1. El escenario más honesto posible

Para que no haya dudas, hemos construido un caso donde **sabemos** que no hay nada que aprender: la
etiqueta es una moneda al aire, independiente de las 1000 columnas de ruido. El mejor modelo posible
aquí debería acertar el 50% (azar puro). Si conseguimos que «acierte» más, será 100% trampa, y podremos
señalar exactamente de dónde viene. Es el experimento clásico de Ambroise y McLachlan que destapó este
error en miles de estudios de genética: cientos de modelos «que predecían el cáncer» no predecían nada,
solo se habían colado mirando el test.


In [ ]:
# ¿De verdad hay columnas que "parecen predictivas" siendo puro ruido? Contemoslas.
corrs = np.array([abs(np.corrcoef(X[:, j], y)[0, 1]) for j in range(1000)])
print(f"Columnas de PURO RUIDO con |correlacion| > 0.20 con la etiqueta: {(corrs > 0.20).sum()} de 1000")
print(f"Columnas con |correlacion| > 0.25: {(corrs > 0.25).sum()}")
print(f"La columna mas 'predictiva' POR AZAR correlaciona {corrs.max():.2f} con la etiqueta.")
print("\nNinguna de estas columnas sabe nada: solo es casualidad. Si eliges 'las mejores' mirando todo")
print("el dataset, te quedas con estas casualidades... y el test ya ha votado que columnas eliges.")


## 2. La trampa: elegir las features mirando *todos* los datos

Con 1000 columnas de ruido, algunas parecen predictivas **por puro azar** (lo acabamos de contar). El
error clásico: seleccionar las «mejores» features usando el dataset **entero** y *después* partir en
train/test. El test ya ha influido en la elección de las columnas: hemos hecho trampa sin darnos cuenta.


In [ ]:
# MAL: seleccionamos las 20 mejores features con TODO el dataset (incluido el test)
X_filtrado = SelectKBest(f_classif, k=20).fit_transform(X, y)
acc_tramposa = cross_val_score(LogisticRegression(max_iter=1000), X_filtrado, y, cv=5).mean()
print(f"Acierto (con fuga):  {acc_tramposa:.3f}   <- 'parece un modelo estupendo' sobre RUIDO")


## 3. Hacerlo bien: el test no se toca hasta el final

La regla de oro: **cualquier** decisión que mire los datos (escalar, seleccionar features, imputar
valores) se aprende **solo con el train**. Metemos la selección de features *dentro* del pipeline de
validación cruzada, para que en cada pliegue el test esté de verdad escondido cuando se eligen las
columnas.


In [ ]:
# BIEN: la seleccion va DENTRO del pipeline; se reaprende solo con el train de cada pliegue
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
acc_honesta = cross_val_score(pipe, X, y, cv=5).mean()
print(f"Acierto (con fuga):    {acc_tramposa:.3f}")
print(f"Acierto (honesto):     {acc_honesta:.3f}   <- cerca de 0.50, el del azar: NO hay nada que aprender")
print(f"\nLa fuga inflaba el resultado en {100*(acc_tramposa-acc_honesta):.0f} puntos, sobre datos sin senal.")
print("Mismo modelo, mismos datos. La unica diferencia: CUANDO miramos el test.")


**Eso es leakage.** Ninguna línea daba error. El primer número era una **buena noticia falsa** que te
habría llevado a desplegar con confianza un modelo que en realidad no aprende nada. La fuga no se caza
con un *try/except*: se caza con disciplina, separando el test *antes* de tocar nada.


## 4. Viéndolo en cámara lenta: un solo split partido a mano

La validación cruzada hace cinco particiones de golpe y puede parecer magia. Vamos a verlo con **un solo
corte**, hecho a mano, para que se toque con el dedo dónde está la trampa. Primero la versión tramposa
(seleccionamos columnas mirando todo y luego partimos) y después la honesta (partimos primero y
seleccionamos columnas mirando solo el train). El test es exactamente el mismo en los dos casos.


In [ ]:
# Tramposo: elegimos las 20 mejores columnas con TODO, y solo despues partimos
sel_todo = SelectKBest(f_classif, k=20).fit(X, y)              # mira X e y ENTEROS (incluye test)
Xt = sel_todo.transform(X)
Xtr, Xte, ytr, yte = train_test_split(Xt, y, test_size=0.3, random_state=1)
m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
acc_mal_1split = (m.predict(Xte) == yte).mean()

# Honesto: partimos PRIMERO; las columnas se eligen mirando solo el train
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=1)
sel_train = SelectKBest(f_classif, k=20).fit(Xtr, ytr)         # mira solo el TRAIN
m = LogisticRegression(max_iter=1000).fit(sel_train.transform(Xtr), ytr)
acc_bien_1split = (m.predict(sel_train.transform(Xte)) == yte).mean()

print(f"Un solo split, columnas elegidas con TODO (mal):   {acc_mal_1split:.3f}")
print(f"Un solo split, columnas elegidas con el train (ok): {acc_bien_1split:.3f}")
print("\nMismo test, mismas 20 columnas candidatas. Solo cambia QUIEN pudo mirar el test al elegirlas.")


## 5. Experimento: cuanto más miras, más te engañas

La fuga por selección de features empeora cuanto **más columnas** tienes, porque con más candidatas es
más fácil encontrar correlaciones espurias por azar. Lo medimos: repetimos la trampa con 200, 500, 2000
y 5000 columnas de ruido y vemos cómo crece el acierto falso (recuerda: el honesto siempre ronda 0,50).


In [ ]:
print("nº de columnas de ruido | acierto CON fuga | acierto honesto")
for n_cols in [200, 500, 2000, 5000]:
    Xn = np.random.normal(size=(120, n_cols)); yn = np.random.randint(0, 2, 120)
    tramposa = cross_val_score(LogisticRegression(max_iter=1000),
                               SelectKBest(f_classif, k=20).fit_transform(Xn, yn), yn, cv=5).mean()
    honesta = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)),
                              Xn, yn, cv=5).mean()
    print(f"       {n_cols:>5}           |      {tramposa:.3f}      |     {honesta:.3f}")
print("\nMas columnas -> mas correlaciones espurias -> mayor acierto FALSO. El honesto se queda en el azar.")


## 6. Segundo disfraz: *target leakage* (la respuesta colada en una columna)

El caso más caro de la vida real no es seleccionar mal: es tener una columna que contiene, directa o
indirectamente, **la propia respuesta**. Imagina que quieres predecir si un cliente devolverá un
producto, y entre tus columnas está «importe ya reembolsado». Esa columna solo tiene valor cuando la
devolución ya ocurrió: el modelo «acierta» porque le estás dando la solución. Lo simulamos: a un
problema con algo de señal real le añadimos una columna que es casi la etiqueta.


In [ ]:
# Problema con algo de senal REAL (no es ruido puro): y depende de verdad de unas columnas
rng = np.random.default_rng(3)
n = 600
Z = rng.normal(size=(n, 8))
y2 = (Z[:,0] + 0.7*Z[:,1] - 0.5*Z[:,2] + rng.normal(0, 0.5, n) > 0).astype(int)

# Modelo honesto: solo con las 8 columnas legitimas
acc_real = cross_val_score(LogisticRegression(max_iter=1000), Z, y2, cv=5).mean()

# Columna tramposa: la respuesta con un poco de ruido encima (un "importe ya reembolsado")
chivata = y2 + rng.normal(0, 0.01, n)
Z_leak = np.column_stack([Z, chivata])
acc_leak = cross_val_score(LogisticRegression(max_iter=1000), Z_leak, y2, cv=5).mean()

print(f"Acierto con columnas legitimas:      {acc_real:.3f}   <- el rendimiento REAL del problema")
print(f"Acierto anadiendo la columna chivata: {acc_leak:.3f}   <- 'casi perfecto', demasiado bonito")
print("\nNingun error salta. Una sola columna que huele a la respuesta dispara el acierto: eso es target leakage.")


### Cómo cazarlo en la práctica

En un dataset real la columna chivata no viene etiquetada como «chivata». Pero deja una huella: su
correlación con la etiqueta es **sospechosamente alta** comparada con el resto. Una primera criba barata
es mirar qué columna correlaciona casi perfectamente con lo que quieres predecir y preguntarse: «¿esto
lo tendré *antes* de conocer la respuesta, o es la respuesta disfrazada?».


In [ ]:
corr_features = [abs(np.corrcoef(Z_leak[:, j], y2)[0, 1]) for j in range(Z_leak.shape[1])]
for j, c in enumerate(corr_features):
    etiqueta = "  <- SOSPECHOSA: demasiado alta" if c > 0.9 else ""
    nombre = f"columna {j}" if j < 8 else f"columna {j} (la chivata)"
    print(f"{nombre:>22}: |correlacion con la etiqueta| = {c:.2f}{etiqueta}")
print("\nLas legitimas correlacionan flojito; la chivata casi 1.0. Esa es la bandera roja que hay que mirar.")


## 7. Tercer disfraz: fuga por escalado

Normalizar (restar la media y dividir por la desviación) parece inofensivo. Pero si calculas esa media y
esa desviación con **todo el dataset** antes de partir, el train ya «sabe» algo del test: la fuga es más
sutil, pero está. Lo comparamos haciéndolo de las dos formas sobre el mismo problema con señal real. El
efecto aquí es **pequeño** (escalar filtra poca información), y eso es justo la lección: la fuga sutil no
se ve en el número, solo se evita con método.


In [ ]:
# MAL: escalamos con TODO el dataset y luego validamos
Xs_mal = StandardScaler().fit_transform(Z)
acc_escala_mal = cross_val_score(LogisticRegression(max_iter=1000), Xs_mal, y2, cv=5).mean()

# BIEN: el escalado va dentro del pipeline, se reaprende en cada pliegue solo con el train
pipe_ok = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
acc_escala_bien = cross_val_score(pipe_ok, Z, y2, cv=5).mean()

print(f"Escalado con todo el dataset (mal):   {acc_escala_mal:.3f}")
print(f"Escalado dentro del pipeline (bien):  {acc_escala_bien:.3f}")
print(f"\nDiferencia: {abs(acc_escala_mal-acc_escala_bien):.4f}. Aqui es minuscula, casi imperceptible.")
print("Por eso la fuga por escalado es traicionera: no la ves en el numero, pero con datasets pequenos")
print("o muchas transformaciones se acumula. La regla no cambia: cada transformacion, solo con el train.")


## 8. Cuarto disfraz: fuga temporal

En datos con fechas (ventas, sensores, clics) el futuro no debe colarse en el pasado. Si mezclas las
fechas al azar para partir train/test, estás entrenando con días posteriores para «predecir» días
anteriores: imposible en producción, donde solo tienes el pasado. Lo montamos con una serie que sube con
el tiempo y comparamos partir **al azar** (mal) frente a partir **por fecha** (bien, train = pasado,
test = futuro).


In [ ]:
# Serie temporal: una tendencia que crece con el tiempo + algo de ruido
t = np.arange(400)
señal = t / 400.0
ruido = rng.normal(0, 0.3, 400)
val = señal + ruido
yt = (val > np.median(val)).astype(int)        # etiqueta: ¿por encima de la mediana?
Xtemp = np.column_stack([t / 400.0, ruido])    # una columna lleva el tiempo

# MAL: barajamos las fechas (el futuro se mezcla con el pasado)
acc_baraja = cross_val_score(LogisticRegression(max_iter=1000), Xtemp, yt, cv=5).mean()

# BIEN: train = primer 70% del tiempo, test = ultimo 30% (como en produccion)
c = 280
m = LogisticRegression(max_iter=1000).fit(Xtemp[:c], yt[:c])
acc_porfecha = (m.predict(Xtemp[c:]) == yt[c:]).mean()

print(f"Partiendo al azar, mezclando fechas (mal):   {acc_baraja:.3f}")
print(f"Partiendo por fecha, train=pasado (bien):    {acc_porfecha:.3f}")
print("\nAl barajar, el modelo ve dias de todo el rango y le resulta facil. Partido por fecha tiene que")
print("extrapolar al futuro, que es lo que de verdad hara en produccion. El primer numero es optimismo.")


## 9. Quinto disfraz: filas duplicadas

Un clásico silencioso: el mismo registro aparece dos veces (por un *join* mal hecho, por reintentos, por
logs repetidos). Si una copia cae en train y otra en test, el modelo «ya ha visto» ese caso: lo borda en
el test sin haber aprendido nada general. Lo demostramos con un vecino más cercano (1-NN), que memoriza:
si en train está el gemelo exacto de un caso de test, lo acierta a distancia cero.


In [ ]:
kf = KFold(5, shuffle=True, random_state=0)
rng2 = np.random.default_rng(7)
nd = 300
Zd = rng2.normal(size=(nd, 5))
yd = (Zd[:,0] - 0.5*Zd[:,1] + rng2.normal(0, 1.2, nd) > 0).astype(int)  # senal moderada, mucho ruido

acc_sin_dup = cross_val_score(KNeighborsClassifier(1), Zd, yd, cv=kf).mean()

# duplicamos cada fila: ahora cada registro tiene un gemelo identico en el dataset
Zdup = np.repeat(Zd, 2, axis=0); ydup = np.repeat(yd, 2)
acc_con_dup = cross_val_score(KNeighborsClassifier(1), Zdup, ydup, cv=kf).mean()

print(f"Sin duplicados (1-NN):  {acc_sin_dup:.3f}   <- el acierto real, modesto")
print(f"Con cada fila duplicada: {acc_con_dup:.3f}   <- disparado de golpe, pura ilusion")
print("\nEl gemelo identico esta en el train cuando su copia cae en el test: el modelo no aprende, recuerda.")
print("Por eso se deduplica ANTES de partir, y se evita que filas relacionadas caigan a ambos lados.")


## 10. Los cinco disfraces, de un vistazo

Todos comparten el mismo pecado de fondo: dejar que el test o el futuro influyan en el entrenamiento.

- **Selección de features con todo el dataset:** eliges columnas mirando el test. Empeora con más columnas.
- **Target leakage:** una columna que es (o contiene) la respuesta. Acierto cercano al perfecto; cázalo
  buscando correlaciones sospechosamente altas.
- **Fuga por escalado/imputación:** calculas medias o rellenos con todo antes de partir. Sutil, casi
  invisible en el número, pero acumulable.
- **Fuga temporal:** barajas fechas y entrenas con el futuro. Se parte por tiempo.
- **Filas duplicadas:** el mismo caso en train y test. Se deduplica antes de partir.

La defensa es siempre la misma: **separa el test primero** y aprende cada transformación solo con el train.


## 11. Pruébalo tú

1. **Sube `k`** en la sección 5 (selecciona 50 o 100 features mirando el test): ¿crece la fuga? Cuantas
   más eliges mirando el test, peor.
2. **Ensucia la chivata:** en la sección 6 cambia `0.01` por `0.3` o `1.0`. ¿A partir de cuánto ruido la
   columna deja de delatarse? Así se «esconde» un *target leakage* sin querer.
3. **Encadena transformaciones:** en la sección 7 mete también un `SelectKBest` mal hecho además del
   escalado. Las fugas pequeñas, sumadas, dejan de ser pequeñas.
4. **Cambia el modelo de la sección 9** a `LogisticRegression`: ¿el duplicado infla menos? Los modelos
   que memorizan (vecinos, árboles profundos) sufren más la fuga por duplicados.
5. **Repite con varias semillas** el experimento 1: el acierto honesto baila alrededor de 0,50; el
   tramposo, siempre por encima.


## 12. Errores comunes

- **Preprocesar antes de partir.** Escalar, seleccionar o imputar con todo el dataset filtra información
  del test. Hazlo dentro de la validación, nunca antes.
- **Un resultado sospechosamente bueno y no desconfiar.** La primera pregunta ante un acierto redondo no
  es «¿qué algoritmo?», sino «¿se me ha colado el futuro o el test?».
- **Confiar en columnas «mágicas».** Una feature demasiado predictiva suele ser *target leakage*
  disfrazado: una variable que en producción no tendrás hasta después de conocer la respuesta.
- **Barajar series temporales.** En datos con fechas, partir al azar mezcla futuro y pasado. Se parte por
  tiempo: train = pasado, test = futuro.
- **No deduplicar.** Filas repetidas a ambos lados del split inflan el acierto sin que aprendas nada.


## 13. Qué te llevas

- El **leakage** es información del test (o del futuro) que se cuela en el entrenamiento. No da error: da
  resultados demasiado buenos para ser verdad.
- La defensa es **separar el test antes de todo** y meter cada decisión (escalar, seleccionar, imputar)
  **dentro** de la validación; en series temporales, partir por fecha; siempre, deduplicar.
- El problema **crece con el número de columnas** y tiene muchos disfraces (selección, *target*, escalado,
  temporal, duplicados); algunos saltan a la vista y otros son casi invisibles, pero todos se evitan con
  la misma disciplina.
- Aprender a **oler la fuga** (desconfiar de lo demasiado bueno) protege más que cualquier algoritmo.

**Para seguir:** el capítulo 3 profundiza en *splits* honestos; el siguiente cuaderno muestra otra forma
de engañarte: mirar solo el promedio y no los subgrupos.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*